<a href="https://colab.research.google.com/github/Edubel77/enrutaia/blob/main/D7_lab_2_2_representaciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Lab 2.2 - Paso 1: BoW frente a TF-IDF (solo palabras)
import hashlib, io, json, urllib.request
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score

BASE = "https://raw.githubusercontent.com/Edubel77/enrutaia/main/data/limpio/"
MD5 = {"train": "cc0ff7796f8e1d1df56862a9f708d521", "validation": "ba68845308293612e717aa7c1a55887c"}
datos = {}
for split, esperado in MD5.items():
    contenido = urllib.request.urlopen(BASE + f"{split}.csv").read()
    assert hashlib.md5(contenido).hexdigest() == esperado, f"¡{split}.csv cambió!"
    datos[split] = pd.read_csv(io.BytesIO(contenido))
train, val = datos["train"], datos["validation"]

representaciones = {
    "BoW (conteos)":        CountVectorizer(ngram_range=(1, 1), min_df=2),
    "TF-IDF (palabras)":    TfidfVectorizer(ngram_range=(1, 1), min_df=2, sublinear_tf=True),
    "TF-IDF (palabras 1-2)": TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
}

resultados = {}
for nombre, vectorizador in representaciones.items():
    modelo = make_pipeline(vectorizador, LogisticRegression(max_iter=2000))
    modelo.fit(train["text"], train["dominio"])
    f1 = f1_score(val["dominio"], modelo.predict(val["text"]), average="macro")
    n_caract = len(modelo[0].vocabulary_)
    resultados[nombre] = {"macro_f1": round(f1, 4), "n_caracteristicas": n_caract}
    print(f"{nombre:<22} | macro-F1: {f1:.4f} | características: {n_caract:,}")

print("\nReferencia E0 (palabras 1-2 + caracteres 2-5): 0.8725")

# ¿Qué palabras tienen MENOS peso en TF-IDF? (las más comunes en el corpus)
tfidf = TfidfVectorizer(min_df=2).fit(train["text"])
idf = pd.Series(tfidf.idf_, index=tfidf.get_feature_names_out()).sort_values()
print("\nPalabras con IDF más bajo (pesan menos):", idf.head(8).round(2).to_dict())
print("Palabras con IDF más alto (pesan más):", idf.tail(5).round(2).to_dict())

BoW (conteos)          | macro-F1: 0.8559 | características: 3,114
TF-IDF (palabras)      | macro-F1: 0.8570 | características: 3,114
TF-IDF (palabras 1-2)  | macro-F1: 0.8499 | características: 10,017

Referencia E0 (palabras 1-2 + caracteres 2-5): 0.8725

Palabras con IDF más bajo (pesan menos): {'de': 1.92, 'la': 2.44, 'el': 2.52, 'en': 2.82, 'mi': 3.06, 'que': 3.11, 'las': 3.17, 'por': 3.22}
Palabras con IDF más alto (pesan más): {'abdominales': 9.22, 'abc': 9.22, 'abascal': 9.22, 'él': 9.22, 'éxito': 9.22}


In [2]:
# Lab 2.2 - Paso 2: vecinos más cercanos en los embeddings de E1
import tensorflow as tf
from tensorflow import keras

tf.keras.utils.set_random_seed(42)
LARGO = 24
dominios = sorted(train["dominio"].unique())
a_id = {d: i for i, d in enumerate(dominios)}

vectorizador = keras.layers.TextVectorization(max_tokens=10_000, output_sequence_length=LARGO)
vectorizador.adapt(train["text"].to_numpy())
vocab = vectorizador.get_vocabulary()
X_tr = vectorizador(tf.constant(train["text"].tolist())).numpy()
X_va = vectorizador(tf.constant(val["text"].tolist())).numpy()

modelo = keras.Sequential([
    keras.Input(shape=(LARGO,), dtype="int64"),
    keras.layers.Embedding(len(vocab), 64, mask_zero=True, name="embedding"),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(len(dominios), activation="softmax"),
])
modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
modelo.fit(X_tr, train["dominio"].map(a_id), validation_data=(X_va, val["dominio"].map(a_id)),
           epochs=50, batch_size=32, verbose=0,
           callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)])

# La matriz de embeddings: una fila de 64 números por palabra
E = modelo.get_layer("embedding").get_weights()[0]
E_norm = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)
indice = {p: i for i, p in enumerate(vocab)}
print("Matriz de embeddings:", E.shape)

def vecinos(palabra, k=6):
    if palabra not in indice:
        return f"'{palabra}' no está en el vocabulario"
    sims = E_norm @ E_norm[indice[palabra]]
    orden = np.argsort(-sims)[1:k + 1]               # el primero es la propia palabra
    return [(vocab[i], round(float(sims[i]), 2)) for i in orden]

for p in ["alarma", "despertador", "música", "lluvia", "correo", "chiste", "luz", "de"]:
    print(f"{p:<12} -> {vecinos(p)}")

Matriz de embeddings: (6589, 64)
alarma       -> [(np.str_('levantarme'), 0.96), (np.str_('enseñas'), 0.96), (np.str_('despertarme'), 0.96), (np.str_('puestas'), 0.96), (np.str_('alarmas'), 0.96), (np.str_('suene'), 0.96)]
despertador  -> [(np.str_('alarmas'), 0.98), (np.str_('levantarme'), 0.98), (np.str_('despertarme'), 0.98), (np.str_('puestas'), 0.98), (np.str_('suene'), 0.97), (np.str_('enseñas'), 0.97)]
música       -> [(np.str_('musica'), 0.96), (np.str_('pandora'), 0.93), (np.str_('canción'), 0.92), (np.str_('rap'), 0.91), (np.str_('reproducción'), 0.91), (np.str_('favorita'), 0.9)]
lluvia       -> [(np.str_('probabilidad'), 0.97), (np.str_('necesitar'), 0.97), (np.str_('ponerme'), 0.97), (np.str_('soleado'), 0.96), (np.str_('lloviendo'), 0.96), (np.str_('hará'), 0.96)]
correo       -> [(np.str_('correos'), 0.99), (np.str_('emails'), 0.98), (np.str_('enviado'), 0.98), (np.str_('gmail'), 0.98), (np.str_('arroba'), 0.98), (np.str_('recibidos'), 0.98)]
chiste       -> [(np.str_('c

In [3]:
# Lab 2.2 - Paso 3: el tokenizador de BETO (WordPiece)
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-uncased")
print("Tamaño del vocabulario de BETO:", tok.vocab_size)

ejemplos = ["pon el despertador a las siete", "borra los despertadores",
            "música de shakira", "musica de shakira", "olly enciende la luz",
            "qué tiempo hará en barranquilla", "enviale un correo a daniel"]
for t in ejemplos:
    print(f"{t:<34} -> {tok.tokenize(t)}")

# Comparación con E1 sobre validación
toks_val = [tok.tokenize(t) for t in val["text"]]
pct_unk = sum(t.count("[UNK]") for t in toks_val) / sum(len(t) for t in toks_val)
largo_tok = np.array([len(t) for t in toks_val])
largo_pal = val["text"].str.split().str.len().to_numpy()
print(f"\nTokens [UNK] en validación con BETO: {pct_unk:.2%}  (E1: 4.7% de palabras desconocidas)")
print(f"Tokens por texto: mediana {np.median(largo_tok):.0f} | percentil 99: {np.percentile(largo_tok, 99):.0f}")
print(f"Palabras por texto: mediana {np.median(largo_pal):.0f} | percentil 99: {np.percentile(largo_pal, 99):.0f}")
print(f"Tokens por palabra (promedio): {largo_tok.sum() / largo_pal.sum():.2f}")

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/310 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/486k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Tamaño del vocabulario de BETO: 31002
pon el despertador a las siete     -> ['pon', 'el', 'despertado', '##r', 'a', 'las', 'siete']
borra los despertadores            -> ['borra', 'los', 'despertado', '##res']
música de shakira                  -> ['música', 'de', 'sha', '##kir', '##a']
musica de shakira                  -> ['musica', 'de', 'sha', '##kir', '##a']
olly enciende la luz               -> ['ol', '##ly', 'enciende', 'la', 'luz']
qué tiempo hará en barranquilla    -> ['qué', 'tiempo', 'hará', 'en', 'barran', '##quilla']
enviale un correo a daniel         -> ['envi', '##ale', 'un', 'correo', 'a', 'daniel']

Tokens [UNK] en validación con BETO: 0.03%  (E1: 4.7% de palabras desconocidas)
Tokens por texto: mediana 7 | percentil 99: 20
Palabras por texto: mediana 7 | percentil 99: 19
Tokens por palabra (promedio): 1.09


In [4]:
# Lab 2.2 - Paso 4: comparativa de representaciones para EnrutaIA
import os
os.makedirs("results", exist_ok=True)

comparativa = {
    "split_evaluado": "validation",
    "metrica": "macro-F1",
    "representaciones": {
        "BoW (palabras)":                      resultados["BoW (conteos)"]["macro_f1"],
        "TF-IDF (palabras)":                   resultados["TF-IDF (palabras)"]["macro_f1"],
        "TF-IDF (palabras 1-2)":               resultados["TF-IDF (palabras 1-2)"]["macro_f1"],
        "E0: TF-IDF palabras 1-2 + caracteres": 0.8725,
        "E0-lemas (spaCy)":                    0.8764,
        "E1: Embedding aprendido (3 semillas)": "0.8732 ± 0.0019",
    },
    "cobertura_vocabulario_validacion": {
        "E1 TextVectorization: palabras desconocidas": "4.7%",
        "BETO WordPiece: tokens [UNK]": f"{pct_unk:.2%}",
    },
    "longitud_en_tokens_BETO": {"mediana": int(np.median(largo_tok)),
                                "percentil_99": int(np.percentile(largo_tok, 99))},
    "decision_max_length_semana3": 32,
}
with open("results/representaciones.json", "w") as f:
    json.dump(comparativa, f, indent=2, ensure_ascii=False)
print(json.dumps(comparativa, indent=2, ensure_ascii=False))

{
  "split_evaluado": "validation",
  "metrica": "macro-F1",
  "representaciones": {
    "BoW (palabras)": 0.8559,
    "TF-IDF (palabras)": 0.857,
    "TF-IDF (palabras 1-2)": 0.8499,
    "E0: TF-IDF palabras 1-2 + caracteres": 0.8725,
    "E0-lemas (spaCy)": 0.8764,
    "E1: Embedding aprendido (3 semillas)": "0.8732 ± 0.0019"
  },
  "cobertura_vocabulario_validacion": {
    "E1 TextVectorization: palabras desconocidas": "4.7%",
    "BETO WordPiece: tokens [UNK]": "0.03%"
  },
  "longitud_en_tokens_BETO": {
    "mediana": 7,
    "percentil_99": 19
  },
  "decision_max_length_semana3": 32
}
